# ショート動画 自動生成（スマホから実行できる）

スマホのブラウザで Colab を開いて、上から順に ▶ を押すだけ。
iPhone / Android どちらでも同じように動く。アプリのインストールは不要。

**やること**
1. 環境をつくる
2. Google Drive をつなぐ（成果物を残すため）
3. Pexels から縦動画を落とす
4. AI画像を生成する
5. ナレーションを作る
6. 動画を組み立てる
7. 確認してダウンロード

3・4・5 は独立しているので、必要なものだけ実行してよい。
**セッションが切れると /content は消える。** Drive に置いた分だけが残る。

---

**APIキーの扱い**：左の 🔑 マーク（シークレット）に登録して使う。
セルに直接ベタ書きするとノートブックを共有したときに漏れる。

## 1. 環境をつくる

1〜2分かかる。セッションごとに1回だけ実行する。

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-noto-cjk > /dev/null
!pip -q install pillow numpy matplotlib

import subprocess, shutil
print('ffmpeg :', shutil.which('ffmpeg'))

# テロップ用の日本語フォント（太字）
FONT = '/usr/share/fonts/opentype/noto/NotoSansCJK-Black.ttc'
import os, glob
if not os.path.exists(FONT):
    cands = glob.glob('/usr/share/fonts/**/NotoSansCJK*', recursive=True)
    FONT = cands[0] if cands else ''
print('フォント :', FONT or '見つからない → 手動で指定が必要')

## 2. Google Drive をつなぐ

認証を求められたらアカウントを選んで許可する。
Drive の `MyDrive/shorts/` に作業フォルダを作る。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJ = '/content/drive/MyDrive/shorts/01_地球の音'
for sub in ('', '素材_画像', '素材_動画', '音声', '完成'):
    os.makedirs(os.path.join(PROJ, sub), exist_ok=True)
os.chdir(PROJ)
print('作業フォルダ :', PROJ)
print(os.listdir(PROJ))

### スクリプトと台本を置く

初回だけ。GitHub から取ってくるか、Drive に手で入れておく。

必要なファイル：
`cuts.csv` / `素材調達リスト.md` / `画像プロンプト_全32点.txt` /
`download_pexels.py` / `generate_rest.py` / `render_assets.py` / `build_video.py` /
`ナレーション原稿.txt` / `voicevox_tts.py`

In [ ]:
REPO = 'https://github.com/masa186/claude-mcp'   # 自分のリポジトリに合わせる
BRANCH = 'claude/prompt-creation-dqzlyr'
SRC = 'prompts/mystery-shorts/assets/01_地球の音'

import subprocess, shutil, os
if not os.path.exists('/content/repo'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO, '/content/repo'],
                   check=True)

src = os.path.join('/content/repo', SRC)
for root, _, files in os.walk(src):
    rel = os.path.relpath(root, src)
    if rel.startswith(('素材_', '音声', '完成', '_作業中')):
        continue
    os.makedirs(os.path.join(PROJ, rel), exist_ok=True)
    for fn in files:
        if fn.endswith(('.py', '.csv', '.txt', '.md')):
            shutil.copy(os.path.join(root, fn), os.path.join(PROJ, rel, fn))

print(sorted(os.listdir(PROJ)))

## 3. Pexels から縦動画を落とす（24カット）

無料キーを https://www.pexels.com/api/ で取得し、
左の 🔑 に `PEXELS_API_KEY` という名前で登録しておく。

1カットにつき2本落とす。1本目が外れなことが多いので、あとで選ぶ。

In [ ]:
from google.colab import userdata
import os
os.environ['PEXELS_API_KEY'] = userdata.get('PEXELS_API_KEY')

!python3 download_pexels.py --dry-run

In [ ]:
# 本番。数分かかる。取得済みはスキップするので、途中で切れても再実行でよい
!python3 download_pexels.py --per-cut 2

## 4. AI画像を生成する（32点）

🔑 に `GEMINI_API_KEY` を登録しておく。Imagen は 9:16 をそのまま出せる。

図版3点（カット6・7・32・34-35用）は生成不要。次のセルで描く。

In [ ]:
# 図版3点。APIキー不要、数秒で終わる
!python3 render_assets.py

In [ ]:
from google.colab import userdata
import os
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

!python3 generate_rest.py --provider gemini --dry-run

In [ ]:
# 本番。生成済みはスキップされる
!python3 generate_rest.py --provider gemini

## 5. ナレーションを作る（41本）

**ここだけ手順が2通りある。**

### 方法A：VOICEVOX ENGINE を Colab 上で動かす
次のセルで Linux版エンジンを落として起動し、`voicevox_tts.py` から叩く。
うまく動けばこれが一番速い（41本が1〜2分）。
**この方法は未検証。** 動かなければ方法Bに切り替える。

### 方法B：VOICEVOX web版でつくって Drive に入れる
スマホのブラウザで web版を開き、`ナレーション原稿.txt` の
各行を読み上げさせて保存する。ファイル名は `cut01.wav` 形式にし、
Drive の `音声/` に入れる（対応表は `ナレーション手順.md`）。
手間はかかるが確実。PCが使えるときにここだけ済ませておくのが現実的。

In [ ]:
# 方法A：VOICEVOX ENGINE を起動する（未検証）
# 失敗したら方法Bへ。ここで止まってもあとの工程には進める
import subprocess, time, urllib.request, os

if not os.path.exists('/content/voicevox_engine'):
    print('リリース一覧: https://github.com/VOICEVOX/voicevox_engine/releases')
    print('Linux CPU版のURLを下の URL に入れて、コメントを外して実行する')
    URL = ''   # 例: https://github.com/VOICEVOX/voicevox_engine/releases/download/<版>/<ファイル>
    if URL:
        !mkdir -p /content/voicevox_engine
        !wget -q -O /content/vv.7z "{URL}"
        !apt-get -qq install -y p7zip-full > /dev/null
        !7z x -y -o/content/voicevox_engine /content/vv.7z > /dev/null

# 起動（バックグラウンド）
import glob
runs = glob.glob('/content/voicevox_engine/**/run', recursive=True)
if runs:
    subprocess.Popen([runs[0], '--host', '127.0.0.1', '--port', '50021'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        try:
            urllib.request.urlopen('http://127.0.0.1:50021/version', timeout=3)
            print('エンジン起動 OK'); break
        except Exception:
            time.sleep(2)
    else:
        print('起動できず → 方法Bへ')
else:
    print('エンジン未配置 → 方法Bへ')

In [ ]:
# 話者IDを調べる（青山龍星を探す）
!python3 voicevox_tts.py --list-speakers

In [ ]:
SPEAKER = 13     # 上で調べたIDに書き換える
SPEED = 1.10

# まず1本だけ。カット10が2.3秒前後になるよう SPEED を調整する
!python3 voicevox_tts.py --speaker {SPEAKER} --speed {SPEED} --only 10 --force

In [ ]:
# 本番。41本。最後に合計尺と推奨SPEEDが出る
!python3 voicevox_tts.py --speaker {SPEAKER} --speed {SPEED}

## 6. 動画を組み立てる

まず `--check` で素材の過不足を見る。足りないカットが出たら、
Pexels の検索ワードを変えるか AI生成に回す。

In [ ]:
!python3 build_video.py --check --font "{FONT}"

In [ ]:
# まず10カットだけ試作して、絵とテロップの相性を見る
!python3 build_video.py --only 1-10 --font "{FONT}" --out /content/preview.mp4

In [ ]:
# 本番。44カット。数分かかる
BGM = ''    # BGMファイルを Drive に置いたらパスを入れる
bgm_arg = f'--bgm "{BGM}" --bgm-db -24' if BGM else ''
!python3 build_video.py --font "{FONT}" {bgm_arg} --out "{PROJ}/地球の音.mp4"

## 7. 確認してダウンロード

In [ ]:
# スマホのブラウザ上で再生して確認する
from IPython.display import HTML
from base64 import b64encode

path = '/content/preview.mp4'      # 本番を見るなら PROJ + '/地球の音.mp4'
data = b64encode(open(path, 'rb').read()).decode()
HTML(f'<video width=320 controls src="data:video/mp4;base64,{data}"></video>')

In [ ]:
# 完成品は Drive に入っているので、Driveアプリから端末に保存できる。
# 直接ダウンロードしたい場合はこちら
from google.colab import files
files.download(f'{PROJ}/地球の音.mp4')

---

## 補足

**スマホで実行するときのコツ**
- 長いセルの実行中に画面を消すとセッションが切れることがある。画面を点けたままにする
- 切れても、成果物は Drive に残っているので途中から再開できる（各スクリプトは取得済みをスキップする）
- 無料枠には実行時間の制限がある。3・4・6 を別々の日に分けても問題ない

**Colab を使わずスマホ本体で回したい場合**
- Android：Termux（F-Droid版）で `pkg install python ffmpeg` → `pip install pillow numpy matplotlib`。
  スクリプトはそのまま動く
- iPhone：a-Shell に Python と ffmpeg は入っているが、Pillow まわりで詰まりやすい。
  素直に Colab を使うほうが早い

**投稿前の確認**
- VOICEVOX はキャラクター個別の規約がある。青山龍星はクレジット表記（`VOICEVOX:青山龍星`）が必要。
  公開前に公式の最新版を必ず自分で確認する
- Pexels素材のライセンス控えは `pexels/credits.csv` に残る
- 台本の `[要確認]` を潰したか